# T10: Malli C1, luokitin YAMNet-upotusten päälle

Tämä notebook ajaa vain koodia, jonka logiikka on `src/yamnet.py`:ssä (`c1_fold_arrays`, `compare_c1`) ja `src/models.py`:ssä (`build_c1`). Sisältö:

1. YAMNet-upotukset välimuistista (T09)
2. Vertailu: pooling (`mean`, `max`, `meanmax`) × piilokerros (0 = lineaarinen, 256), jokainen viidellä ajolla
3. Valinta mallille C1 (`C1_POOLING`, `C1_HIDDEN`)

**Vain opetus- ja validointifoldit.** `compare_c1` heittää testifoldin pois eikä arvioi sitä. Standardoinnin tilastot lasketaan vain opetusfoldeista. Lopulliset testiajot ja `results/C1_fold{k}.json` tehdään T15:ssä.

In [ ]:
import sys
from pathlib import Path

# Repon juuri polkuun, jotta `src` löytyy (notebook ajetaan kansiosta notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import keras

from src.data import load_metadata
from src.yamnet import C1_HIDDEN, C1_POOLING, build_yamnet_cache, compare_c1

# Colabissa: DATA_DIR = "/content/drive/MyDrive/Elainaanet-CNN/data"
DATA_DIR = ROOT / "data"
keras.utils.set_random_seed(42)

df = load_metadata(DATA_DIR)
emb = build_yamnet_cache(df, Path(DATA_DIR) / "cache")    # T09:n välimuisti (400, 10, 1024)
emb.shape

## 2. Vertailu validointifoldeilla

6 asetusta × 5 ajoa = 30 koulutusta. Jokaisen ajon validointitarkkuus lasketaan parhaan epookin painoilla (EarlyStopping, `val_loss`). Tulos tallennetaan tiedostoon `results/T10_c1_vertailu.csv` raporttia ja T18:aa varten.

In [ ]:
res = compare_c1(df, emb)
res.to_csv(ROOT / "results" / "T10_c1_vertailu.csv", index=False)
res.pivot_table(index=["pooling", "hidden"], columns="fold", values="val_acc").round(3)

In [ ]:
summary = (res.groupby(["pooling", "hidden"])
              .agg(val_acc=("val_acc", "mean"), sd=("val_acc", "std"),
                   val_loss=("val_loss", "mean"), epochs=("epochs", "mean"))
              .sort_values("val_acc", ascending=False))
summary.round(3)

In [ ]:
# Validointitarkkuus keskiarvo ± keskihajonta (5 ajoa), järjestettynä. Yksi sarja -> yksi väri.
BLUE, TEXT, MUTED, GRID = "#257de0", "#191919", "#666666", "#e5e5e5"
s = summary.iloc[::-1]                                     # paras ylimmäksi
labels = [f"{p} + {'lineaarinen' if h == 0 else f'Dense {h}'}" for p, h in s.index]
y = range(len(s))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.errorbar(s["val_acc"], y, xerr=s["sd"], fmt="o", color=BLUE, ecolor=BLUE,
            elinewidth=2, capsize=0, markersize=8)
ax.set_yticks(list(y), labels, color=TEXT)
ax.set_xlabel("Validointitarkkuus (5 ajon keskiarvo ± keskihajonta)", color=MUTED)
ax.set_title("C1: pooling ja piilokerros, vain validointifoldit", color=TEXT, loc="left")
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
chosen = labels.index(f"{C1_POOLING} + {'lineaarinen' if C1_HIDDEN == 0 else f'Dense {C1_HIDDEN}'}")
ax.annotate("valittu", (s["val_acc"].iloc[chosen] + s["sd"].iloc[chosen], chosen),
            xytext=(8, 0), textcoords="offset points", va="center", color=TEXT)
plt.tight_layout()
plt.show()

## 3. Valinta

- **Pooling: `max`.** Paras molemmilla malleilla. Keskiarvo laimenee hiljaisista kehyksistä (YAMNet tunnisti monessa leikkeessä "Silence", ks. T09), maksimi poimii eläimen äänen.
- **Piilokerros: ei (`hidden=0`, lineaarinen luokitin).** Ero `max + Dense 256`:een on paljon keskihajontaa pienempi (validointifoldissa 80 leikettä, yksi leike = 1,25 %-yks.), lineaarisella on pienempi validointihäviö ja noin 25 kertaa vähemmän parametreja.

Valinta on vakioina `src/yamnet.py`:ssä (`C1_POOLING`, `C1_HIDDEN`), ja T15 käyttää niitä. Luvut ovat validointituloksia, eivät C1:n testitulos.